# loveAI Persona Model — Kaggle Training

Fine-tunes `google/gemma-3-1b-it` (gated — needs `HF_TOKEN` secret + license acceptance) via QLoRA into the loveAI companion voice: **SFT → DPO → merge → smoke test**.

**Mirrors the repo pipeline** (`ml/train.py`, `ml/train_dpo.py`, `ml/merge.py`, `ml/serve.py`). Outputs a **merged model** in `/kaggle/working/persona-merged` ready for GGUF export / HF push / self-hosting.

## Inputs

| Input | Required | What it provides |
|---|---|---|
| `loveai-persona-data` (this repo's bundle) | **yes** | SFT/DPO JSONL, `persona_prompts.jsonl`, `persona-matrix.json`, `dataset_report.json` |
| `nbertagnolli/counsel-chat` | optional | 2,775 licensed-therapist Q/A pairs |
| `thedevastator/amod-mental-health-counseling-conversations-data` *or* `melissamonfared/mental-health-counseling-conversations-k` | optional | counseling Q/A (Kaggle mirror of the Amod set) |
| `antengage/empathy-conversation-dataset` | optional | 4,008 multi-turn empathy conversations (grief, burnout, loneliness, parenting) |
| `asifahnafchowdhury/empathetic-dialogues-dataset-by-rashkin-cleaned` | optional | cleaned EmpatheticDialogues (24.8k conversations, 32 emotions) |

The optional inputs are merged into the **SFT** set by the second cell block (schema-sniffing adapters mirror `ml/datasets/real_sources.py`, same quality gate). Missing inputs are skipped — the notebook runs fine with just the bundle.

- **Runtime**: GPU T4 x2 or P100, **Internet ON** (model + package downloads)
- **Time**: ~10-20 min for the bundled synthetic set; more with extra inputs (30 GPU-hr/wk free)
- **Licenses**: the extra datasets are research/non-commercial in several cases — that only matters if the resulting model ships in a paid product. See `kaggle/README.md`.

In [ ]:
import os
os.environ['CUDA_VISIBLE_DEVICES'] = '0'   # force single GPU (MUST be set before torch import)

# Install the training stack (matches ml/requirements.txt). Kaggle's base image
# does not ship trl/peft/bitsandbytes — Internet must be ON for this cell.
import subprocess, sys
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q',
    'transformers>=5.0,<6', 'peft>=0.20', 'trl>=1.9', 'datasets>=5.0',
    'accelerate>=1.14', 'bitsandbytes>=0.50'])
print('install OK')

import random
import numpy as np
import torch, transformers, datasets as _ds
print('torch', torch.__version__, '| transformers', transformers.__version__, '| datasets', _ds.__version__,
      '| cuda', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else '')
random.seed(42); np.random.seed(42); torch.manual_seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(42)
print('seeds set (42)')


In [ ]:
import os
from kaggle_secrets import UserSecretsClient
os.environ['HF_TOKEN'] = ''
try:
    _secret = UserSecretsClient().get_secret('HF_TOKEN')
    if _secret:
        os.environ['HF_TOKEN'] = _secret
except Exception:
    pass
print('HF_TOKEN configured successfully.')


In [ ]:
# --- setup: paths, constants, helpers --------------------------------------
import glob
import json
import os
import re
from pathlib import Path

os.environ['CUDA_VISIBLE_DEVICES'] = '0'   # force single GPU (avoids DataParallel split across T4 x2)
BASE_MODEL = 'google/gemma-3-1b-it'        # gated -> HF_TOKEN (cell above)
WORK = Path('/kaggle/working')
WORK.mkdir(parents=True, exist_ok=True)


def _find_data_dir():
    """Prefer the uploaded dataset bundle, fall back to a local checkout."""
    candidates = [
        '/kaggle/input/loveai-persona-data',
        '/kaggle/input/loveai-persona-data/loveai-persona-data',
        *sorted(glob.glob('/kaggle/input/*/persona_sft_train.jsonl')),
        *sorted(glob.glob('/kaggle/input/*/*/persona_sft_train.jsonl')),
        'ml/data',
        '../ml/data',
    ]
    for c in candidates:
        p = Path(c)
        if p.is_file() and p.name == 'persona_sft_train.jsonl':
            return p.parent
        if (p / 'persona_sft_train.jsonl').exists():
            return p
    raise FileNotFoundError(
        'persona_sft_train.jsonl not found. Add the loveAI dataset bundle under '
        'Add Input (slug: loveai-persona-data) or place ml/data next to the notebook.'
    )


DATA_DIR = _find_data_dir()
print('DATA_DIR =', DATA_DIR)
print('files   =', sorted(p.name for p in DATA_DIR.iterdir()))


def load_rows(path):
    return [json.loads(l) for l in Path(path).read_text(encoding='utf-8').splitlines() if l.strip()]


def dump_rows(rows, path):
    with open(path, 'w', encoding='utf-8') as f:
        for row in rows:
            f.write(json.dumps(row) + '\n')
    print(f'wrote {len(rows)} rows -> {path}')


persona_pool = {}
if (DATA_DIR / 'persona_prompts.jsonl').exists():
    persona_pool = {r['persona_id']: r['persona_text'] for r in load_rows(DATA_DIR / 'persona_prompts.jsonl')}
else:
    # Older bundle without the persona pool: lift the system prompts out of the SFT file.
    for row in load_rows(DATA_DIR / 'persona_sft_train.jsonl'):
        persona_pool[row['persona_id']] = row['messages'][0]['content']
print(f'persona prompt pool: {len(persona_pool)} profiles')

## 1. Load the persona matrix + dataset

In [ ]:
import torch
from datasets import Dataset
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from trl import SFTTrainer, SFTConfig, DPOTrainer, DPOConfig

def load_rows(path):
    return [json.loads(l) for l in Path(path).read_text(encoding='utf-8').splitlines() if l.strip()]

def compute_dtype():
    # bf16 only where the GPU has *native* bf16 (Ampere+). On T4 (sm_75) the bare
    # torch.cuda.is_bf16_supported() returns True because it includes emulation,
    # which would train in slow/emulated bf16 -> use including_emulation=False.
    if not torch.cuda.is_available():
        return torch.float32
    try:
        native_bf16 = torch.cuda.is_bf16_supported(including_emulation=False)
    except TypeError:  # torch < 2.3
        native_bf16 = torch.cuda.get_device_capability()[0] >= 8
    return torch.bfloat16 if native_bf16 else torch.float16

def restore_fp32_adapters(trainer):
    # TRL casts the adapter tensors of a *quantized* model to bf16 (QLoRA paper, sect. 3).
    # On T4 (sm_75) there is no native bf16 and torch's fp16 AMP GradScaler refuses to unscale
    # bf16 grads, so put the fp32 master weights back before training starts.
    restored = 0
    for p in trainer.model.parameters():
        if p.requires_grad and p.dtype == torch.bfloat16:
            p.data = p.data.to(torch.float32)
            restored += 1
    if restored:
        print(f'fp16 path: restored {restored} adapter tensors to fp32')

sft_train = load_rows(DATA_DIR / 'persona_sft_train.jsonl')
sft_val   = load_rows(DATA_DIR / 'persona_sft_val.jsonl')
dpo_train = load_rows(DATA_DIR / 'persona_dpo_train.jsonl')
dpo_val   = load_rows(DATA_DIR / 'persona_dpo_val.jsonl')
print(f'SFT train={len(sft_train)} val={len(sft_val)} | DPO train={len(dpo_train)} val={len(dpo_val)}')
print('Sample SFT record:', json.dumps(sft_train[0], indent=2)[:600])
if (DATA_DIR / 'dataset_report.json').exists():
    _report = json.loads((DATA_DIR / 'dataset_report.json').read_text())
    print('report: personas=%s prompts=%s sft=%s dpo=%s real=%s' % (
        _report['personas'], _report['user_prompts'], _report['sft']['total'],
        _report['dpo']['total'], _report['sft']['real']))

In [ ]:
# --- optional: merge extra Kaggle dataset inputs into the SFT set -----------
# Anything present under /kaggle/input is sniffed for conversational columns and
# converted to the loveAI row schema (system persona + conversation).
# Field mapping + quality gate mirror ml/datasets/real_sources.py and
# ml/datasets/build_dataset.py, so the merged rows are indistinguishable from
# bundle rows to train.py. SFT only: the persona-contrasted DPO generator lives
# in ml/datasets/templates.py (run build_dataset.py --real-data in the repo if
# you want real-data DPO pairs baked in).
import csv
import random

MAX_EXTRA_SFT_ROWS = 4000     # cap so synthetic voice data is never swamped
EXTRA_SEED = 42

# Same thresholds + term lists as ml/datasets/build_dataset.py - loveAI is a
# companion, not a crisis service or a clinic, and we do not want it imitating
# either register.
MIN_USER_WORDS, MIN_ASSISTANT_WORDS, MIN_DIALOGUE_WORDS = 2, 4, 12
MAX_WORDS, MAX_ROW_WORDS = 220, 700

# The crisis/clinical term lists below are copied verbatim from
# ml/datasets/build_dataset.py.
CRISIS_TERMS = (
    'suicide', 'suicidal', 'kill myself', 'kill yourself', 'self-harm', 'self harm',
    'cut myself', 'end my life', 'want to die', 'overdose', '988', 'crisis line',
    'emergency services', 'call 911',
)
CLINICAL_TERMS = (
    'as an ai', 'i am an ai', "i'm an ai", 'language model', 'i am not a therapist',
    "i'm not a therapist", 'licensed therapist', 'seek professional help',
    'seek help from a professional', 'consult a professional', 'speak to a professional',
    'medical advice', 'prescrib', 'medication', 'diagnos', 'psychiatr', 'dsm-', 'dsm 5',
    'intake form', 'insurance', 'copay', 'hotline', 'helpline',
)

_WS = re.compile(r'\s+')


def _clean(value):
    return _WS.sub(' ', str(value).replace('\u00a0', ' ')).strip() if value is not None else ''


def _words(text):
    return len(re.findall(r"[A-Za-z']+", text))


def _length_ok(text, min_words=MIN_USER_WORDS, max_words=MAX_WORDS):
    return min_words <= _words(text) <= max_words


def _register_ok(text):
    return not any(t in text.lower() for t in CLINICAL_TERMS)


def _gate_dialogue(turns):
    """Dialogue level: usable at all? (one weak turn must not kill a good chat)"""
    if len(turns) < 2 or turns[0][0] != 'user':
        return False
    blob = ' '.join(c for _, c in turns).lower()
    if any(t in blob for t in CRISIS_TERMS):
        return False
    if sum(_words(c) for _, c in turns) < MIN_DIALOGUE_WORDS:
        return False
    return all(_words(c) <= MAX_WORDS for _, c in turns)


def _gate_row(messages):
    """Row level: exactly what gets trained on."""
    if sum(_words(m['content']) for m in messages) > MAX_ROW_WORDS:
        return False
    for m in messages:
        role, content = m['role'], m['content']
        if role == 'system':
            continue
        minimum = MIN_ASSISTANT_WORDS if role == 'assistant' else MIN_USER_WORDS
        if not _length_ok(content, minimum):
            return False
        if any(t in content.lower() for t in CRISIS_TERMS):
            return False
        if role == 'assistant' and not _register_ok(content):
            return False
    return True


def _read_table(path):
    """Read .jsonl / .json / .csv into a list of dicts (no pandas dependency)."""
    suffix = path.suffix.lower()
    if suffix in ('.jsonl', '.ndjson'):
        return load_rows(path)
    if suffix == '.json':
        data = json.loads(Path(path).read_text(encoding='utf-8'))
        if isinstance(data, dict):
            for key in ('data', 'rows', 'conversations', 'train'):
                if isinstance(data.get(key), list):
                    data = data[key]
                    break
            else:
                data = [data]
        return [d for d in data if isinstance(d, dict)]
    if suffix == '.csv':
        with open(path, newline='', encoding='utf-8', errors='replace') as f:
            return list(csv.DictReader(f))
    return []


_ROLE_KEYS = ('role', 'speaker', 'from', 'author')
_TEXT_KEYS = ('content', 'text', 'utterance', 'message', 'value')
_QA_PAIRS = (
    (('questiontext', 'question', 'context', 'input', 'instruction', 'prompt', 'post', 'questiontitle'),
     ('answertext', 'answer', 'response', 'output', 'comment_1', 'comment', 'reply')),
)


def sniff_dialogues(records):
    """Turns/conversations out of whatever shape the dataset happens to use."""
    out = []
    for rec in records:
        if not isinstance(rec, dict):
            continue
        lowered = {str(k).lower(): v for k, v in rec.items()}

        # 1) already chat-formatted: messages / conversations / dialog
        for key in ('messages', 'conversations', 'conversation', 'dialog', 'dialogue'):
            value = lowered.get(key)
            if isinstance(value, list) and value and isinstance(value[0], dict):
                turns = []
                for turn in value:
                    role = next((turn[k] for k in turn if str(k).lower() in _ROLE_KEYS), '')
                    text = next((turn[k] for k in turn if str(k).lower() in _TEXT_KEYS), '')
                    role = str(role).lower()
                    turns.append(('user' if role in ('user', 'usr', 'seeker', 'patient', 'human', 'speaker_0', '0') else 'assistant', _clean(text)))
                if len(turns) >= 2:
                    out.append(turns)
                break
        else:
            # 2) plain list of utterances -> alternate speakers
            flat = next((lowered[k] for k in ('dialogue', 'utterances', 'conv') if isinstance(lowered.get(k), list) and lowered[k] and isinstance(lowered[k][0], str)), None)
            if flat:
                out.append([('user' if i % 2 == 0 else 'assistant', _clean(u)) for i, u in enumerate(flat)])
                continue

            # 3) Q/A columns
            for q_keys, a_keys in _QA_PAIRS:
                question = next((_clean(lowered[k]) for k in q_keys if _clean(lowered.get(k))), '')
                answer = next((_clean(lowered[k]) for k in a_keys if _clean(lowered.get(k))), '')
                if question and answer:
                    out.append([('user', question), ('assistant', answer)])
                    break
    return out


def normalise(dialogues, source):
    """Gate, merge consecutive same-role turns, drop non-conversations."""
    kept = []
    for turns in dialogues:
        merged = []
        for role, content in turns:
            if not content:
                continue
            if merged and merged[-1][0] == role:
                merged[-1] = (role, f'{merged[-1][1]} {content}')
            else:
                merged.append((role, content))
        if not _gate_dialogue(merged):
            continue
        kept.append((source, merged))
    return kept


def to_sft_rows(tagged, rng, max_messages=6):
    rows, seen = [], set()
    persona_ids = sorted(persona_pool)
    for source, turns in tagged:
        persona_id = persona_ids[rng.randrange(len(persona_ids))]
        for i, (role, content) in enumerate(turns):
            if role != 'assistant' or i == 0:
                continue
            start = max(0, i - max_messages)
            while start < i and turns[start][0] != 'user':
                start += 1
            context = turns[start:i]
            if not context or context[0][0] != 'user':
                continue
            key = content[:200].lower()
            if key in seen:
                continue
            messages = [
                {'role': 'system', 'content': persona_pool[persona_id]},
                *[{'role': r, 'content': c} for r, c in context],
                {'role': 'assistant', 'content': content},
            ]
            if not _gate_row(messages):
                continue
            seen.add(key)
            rows.append({'persona_id': persona_id, 'messages': messages})
    return rows


extra_sft, extra_sources = [], {}
for input_dir in sorted(glob.glob('/kaggle/input/*')):
    slug = Path(input_dir).name
    if slug == Path(DATA_DIR).name:
        continue
    files = [p for p in Path(input_dir).rglob('*') if p.suffix.lower() in ('.jsonl', '.ndjson', '.json', '.csv')]
    records = [rec for p in files for rec in _read_table(p)]
    if not records:
        continue
    tagged = normalise(sniff_dialogues(records), slug)
    if not tagged:
        print(f'  {slug}: {len(records)} records, no conversational columns detected -> skipped')
        continue
    extra_sft.extend(tagged)
    extra_sources[slug] = len(tagged)
    print(f'  {slug}: {len(records)} records -> {len(tagged)} usable dialogues')

if extra_sft:
    rng = random.Random(EXTRA_SEED)
    extra_sft.sort(key=lambda pair: (pair[0], pair[1][0][1][:80]))
    if len(extra_sft) > MAX_EXTRA_SFT_ROWS:
        extra_sft = rng.sample(extra_sft, MAX_EXTRA_SFT_ROWS)
    extra_rows = to_sft_rows(extra_sft, rng)
    rng.shuffle(extra_rows)
    cut = max(1, len(extra_rows) // 10)
    sft_train = sft_train + extra_rows[cut:]
    sft_val = sft_val + extra_rows[:cut]
    print(f'merged {len(extra_rows)} extra SFT rows from {extra_sources}')
    print(f'SFT train={len(sft_train)} val={len(sft_val)}')
else:
    print('no extra dataset inputs found - training on the loveAI bundle only')

## 2. SFT — teach the base model the companion voice

In [ ]:
# Skip SFT if a pre-trained adapter is available as a dataset input
import shutil, glob

def _find_sft_adapter():
    for c in [
        '/kaggle/input/loveai-persona-sft/persona-sft',
        '/kaggle/input/loveai-persona-sft',
        *sorted(glob.glob('/kaggle/input/*/persona-sft')),
        *sorted(glob.glob('/kaggle/input/*/*/adapter_model.safetensors')),
        *sorted(glob.glob('/kaggle/input/*/adapter_model.safetensors')),
    ]:
        p = Path(c)
        if p.is_file() and p.name == 'adapter_model.safetensors':
            return p.parent
        if (p / 'adapter_model.safetensors').exists():
            return p
    return None

sft_found = _find_sft_adapter()
skip_sft = bool(sft_found)
if skip_sft:
    (WORK/'persona-sft').mkdir(parents=True, exist_ok=True)
    for f in sft_found.iterdir():
        if f.is_file():
            shutil.copy2(f, WORK/'persona-sft'/f.name)
    print(f'SFT adapter loaded from {sft_found} — skipping SFT training')
else:
    print('No SFT adapter found — training SFT from scratch')

bnb = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=compute_dtype(),
    bnb_4bit_quant_storage=compute_dtype(),
    bnb_4bit_use_double_quant=True,
)
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL, quantization_config=bnb, device_map={'': 0}, torch_dtype=torch.float16, trust_remote_code=True
)
model = prepare_model_for_kbit_training(model)
lora = LoraConfig(
    r=16, lora_alpha=32, lora_dropout=0.05, bias='none', task_type='CAUSAL_LM',
    target_modules=['q_proj','k_proj','v_proj','o_proj','gate_proj','up_proj','down_proj'],
)
model = get_peft_model(model, lora)
model.print_trainable_parameters()

def format_chat(row):
    return tokenizer.apply_chat_template(row['messages'], tokenize=False, add_generation_prompt=False)

train_ds = Dataset.from_list(sft_train).map(lambda r: {'text': format_chat(r)})
val_ds   = Dataset.from_list(sft_val).map(lambda r: {'text': format_chat(r)})

sft_config = SFTConfig(
    output_dir=str(WORK/'persona-sft'),
    num_train_epochs=3, per_device_train_batch_size=1, per_device_eval_batch_size=1,
    gradient_accumulation_steps=8, learning_rate=2e-4, warmup_steps=20,
    logging_steps=10, eval_strategy='steps', eval_steps=200, save_strategy='epoch',
    bf16=compute_dtype()==torch.bfloat16, fp16=compute_dtype()==torch.float16,
    max_length=512, report_to=[], remove_unused_columns=False, dataset_text_field='text',
    seed=42,
)
if not skip_sft:
    trainer = SFTTrainer(model=model, args=sft_config, train_dataset=train_ds, eval_dataset=val_ds, processing_class=tokenizer)
    restore_fp32_adapters(trainer)
    trainer.train()
    trainer.save_model(str(WORK/'persona-sft'))
    tokenizer.save_pretrained(str(WORK/'persona-sft'))
    print('SFT done ->', WORK/'persona-sft')


## 3. DPO — make it *prefer* persona-aligned replies

In [ ]:
def to_prompt(messages):
    return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

def to_response(messages):
    if isinstance(messages, list) and messages and 'content' in messages[0]:
        content = messages[0]['content']
    else:
        content = str(messages)
    return content + '<end_of_turn>\n'

def build_dpo_ds(rows):
    ds = Dataset.from_list(rows)
    return ds.map(lambda r: {
        'prompt': to_prompt(r['prompt']),
        'chosen': to_response(r['chosen']),
        'rejected': to_response(r['rejected']),
    }, remove_columns=['persona_id','prompt','chosen','rejected'])

# Continue from the SFT adapter — load base model + attach adapter
from trl import DPOTrainer, DPOConfig
from peft import PeftModel
sft_adapter_path = _find_sft_adapter() if '_find_sft_adapter' in globals() else None
if not sft_adapter_path or not (sft_adapter_path / 'adapter_model.safetensors').exists():
    sft_adapter_path = WORK/'persona-sft'

dpo_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL, quantization_config=bnb, device_map={'': 0}, torch_dtype=torch.float16, trust_remote_code=True
)
dpo_model = PeftModel.from_pretrained(dpo_model, str(sft_adapter_path), is_trainable=True)
print('SFT adapter loaded (trainable) from', sft_adapter_path)
dpo_train_ds = build_dpo_ds(dpo_train)
dpo_val_ds   = build_dpo_ds(dpo_val)

dpo_config = DPOConfig(
    output_dir=str(WORK/'persona-dpo'),
    num_train_epochs=2, per_device_train_batch_size=1, per_device_eval_batch_size=1,
    gradient_accumulation_steps=8, learning_rate=1e-4, beta=0.1, warmup_steps=20,
    logging_steps=10, eval_strategy='steps', eval_steps=200, save_strategy='epoch',
    bf16=compute_dtype()==torch.bfloat16, fp16=compute_dtype()==torch.float16,
    max_length=512, report_to=[], seed=42,
)
dpo_trainer = DPOTrainer(
    model=dpo_model, ref_model=None, args=dpo_config,
    train_dataset=dpo_train_ds, eval_dataset=dpo_val_ds,
    peft_config=None, processing_class=tokenizer,
)
restore_fp32_adapters(dpo_trainer)
dpo_trainer.train()
dpo_trainer.save_model(str(WORK/'persona-dpo'))
tokenizer.save_pretrained(str(WORK/'persona-dpo'))
print('DPO done ->', WORK/'persona-dpo')


## 4. Merge the LoRA into a standalone model

In [ ]:
from peft import PeftModel

base = AutoModelForCausalLM.from_pretrained(BASE_MODEL, torch_dtype=torch.bfloat16, device_map='cpu', trust_remote_code=True)
merged = PeftModel.from_pretrained(base, str(WORK/'persona-dpo'))
merged = merged.merge_and_unload()

out = WORK/'persona-merged'
out.mkdir(parents=True, exist_ok=True)
merged.save_pretrained(out, safe_serialization=True)
tokenizer.save_pretrained(out)
size_gb = sum(f.stat().st_size for f in out.glob('*')) / 1e9
print(f'Merged model saved to {out} ({size_gb:.2f} GB)')
print('Files:', sorted(p.name for p in out.iterdir()))

## 5. Smoke test + export a sample reply

In [ ]:
import torch

test = AutoModelForCausalLM.from_pretrained(str(WORK/'persona-merged'), torch_dtype=torch.float16, device_map='auto', trust_remote_code=True)
tok = AutoTokenizer.from_pretrained(str(WORK/'persona-merged'), trust_remote_code=True)

_probe_id = sorted(persona_pool)[0]
messages = [
    {'role': 'system', 'content': persona_pool[_probe_id]},
    {'role': 'user', 'content': 'I had a long day and I am feeling drained.'},
]
text = tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
inputs = tok([text], return_tensors='pt').to(test.device)
with torch.no_grad():
    out = test.generate(**inputs, max_new_tokens=220, temperature=0.8, do_sample=True, top_p=0.95, pad_token_id=tok.eos_token_id)
reply = tok.decode(out[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True).strip()
print('=== SAMPLE REPLY ===')
print(reply)
print('====================')
with open(WORK/'sample-reply.txt', 'w') as f:
    f.write(reply)
print('Saved sample-reply.txt')

## Done

Download `/kaggle/working/persona-merged/` + `sample-reply.txt` from the Kaggle output panel.

Next: GGUF export (`llama.cpp convert_hf_to_gguf.py`) → HF CPU Space / AWS free tier → `LLM_URL` in Vercel.